# Demos: Lecture 14

In [5]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from lecture14_helpers import *

## Demo 1: Modular arithmetic

Demo code adapted from [PennyLane documention of `qml.Multiplier`](https://docs.pennylane.ai/en/stable/code/api/pennylane.Multiplier.html).

In [6]:
x = 3
k = 4
N = 7

x_wires = [0,1,2]
work_wires = [3,4,5,6,7]

dev = qml.device("default.qubit", shots=1)

@qml.qnode(dev)
def circuit():
    qml.BasisEmbedding(x, wires=x_wires)
    #qml.Hadamard(wires=x_wires[0]) # Superposition of |3> = |011> and |7> = |111>
    qml.Multiplier(k, x_wires, N, work_wires)
    qml.Adder(1, x_wires, N, work_wires[:2])
    return qml.sample(wires=x_wires)

In [7]:
circuit()

array([1, 1, 0])

## Demo 2: Order finding

<img src="fig/qpe_full.png" width="800px">

In [8]:
N = 7
a = 5

In [9]:
num_target_qubits = int(np.floor(np.log2(N))) + 1

wires = qml.registers({
    "estimation_wires": 6,
    "target_wires": num_target_qubits,
    "work_wires": num_target_qubits + 2
})

dev = qml.device('default.qubit', shots=1000)

In [10]:
@qml.qnode(dev)
def find_order():

    
    return qml.counts(wires=estimation_wires)

In [ ]:
counts = find_order()
samples = np.array(list(counts.keys()))
frequencies = np.array(list(counts.values()))

In [ ]:
plt.bar([int(c, 2) for c in samples], frequencies)

In [ ]:
possible_r = []
most_common = samples[np.argsort(frequencies)[::-1]]
sorted_frequencies = frequencies[np.argsort(frequencies)[::-1]]

for sample in most_common:
    phase = fractional_binary_to_float(sample)
    est_r = phase_to_order(phase, N)
    possible_r.append(est_r)

In [ ]:
for r in range(1, N):
    print(f"r = {r}: {np.sum(sorted_frequencies[np.where(np.array(possible_r) == r)]) / dev.shots.total_shots}")